# Tourism — Travel Demand and Destination Intelligence
India-focused Databricks + PySpark + Delta Lake + Power BI project.

**Important:** sample CSVs are synthetic. Replace them with verified official data.


## 00 Config And Inspection


In [ ]:
# Databricks notebook source

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "tourism_analytics"
VOLUME = "tourism_raw"
BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
TABLE_BASE = f"{CATALOG}.{SCHEMA}"

VISITS_PATH = f"{BASE_PATH}/tourism_visits.csv"
WEATHER_PATH = f"{BASE_PATH}/weather_daily.csv"
DESTINATIONS_PATH = f"{BASE_PATH}/destinations.csv"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

def read_csv(path):
    return (spark.read.option("header", True)
            .option("inferSchema", True)
            .option("mode", "PERMISSIVE")
            .csv(path))

visits_raw = read_csv(VISITS_PATH)
weather_raw = read_csv(WEATHER_PATH)
destinations_raw = read_csv(DESTINATIONS_PATH)

for name, df in [
    ("TOURISM VISITS", visits_raw),
    ("WEATHER", weather_raw),
    ("DESTINATIONS", destinations_raw)
]:
    print(f"\n{name}")
    df.printSchema()
    print("Rows:", df.count())
    display(df.limit(10))


## 01 Bronze Ingestion


In [ ]:
# Databricks notebook source

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "tourism_analytics"
TABLE_BASE = f"{CATALOG}.{SCHEMA}"

def write_bronze(df, table_name, source_name):
    output = (df.withColumn("_ingested_at", F.current_timestamp())
                .withColumn("_source_name", F.lit(source_name)))
    (output.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{TABLE_BASE}.{table_name}"))
    print(f"Saved {table_name}: {output.count()} rows")

write_bronze(visits_raw, "bronze_tourism_visits", "tourism_visits.csv")
write_bronze(weather_raw, "bronze_weather", "weather_daily.csv")
write_bronze(destinations_raw, "bronze_destinations", "destinations.csv")


## 02 Silver Transform


In [ ]:
# Databricks notebook source

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "tourism_analytics"
TABLE_BASE = f"{CATALOG}.{SCHEMA}"

COLUMN_MAP = {
    "visits": {
        "date": "date", "state": "state", "destination": "destination",
        "visitor_type": "visitor_type", "visitor_count": "visitor_count"
    },
    "weather": {
        "date": "date", "state": "state", "temp_max": "temp_max_c",
        "temp_min": "temp_min_c", "precipitation": "precipitation_mm"
    },
    "destinations": {
        "destination": "destination", "state": "state",
        "latitude": "latitude", "longitude": "longitude",
        "category": "category"
    }
}

def require_columns(df, mapping, label):
    missing = [source_col for source_col in mapping.values()
               if source_col not in df.columns]
    if missing:
        raise ValueError(
            f"{label}: missing columns {missing}. "
            f"Available columns: {df.columns}. Update COLUMN_MAP."
        )

v = COLUMN_MAP["visits"]
require_columns(visits_raw, v, "visits")
visits = (visits_raw.select(
    F.to_date(F.col(v["date"])).alias("visit_date"),
    F.initcap(F.trim(F.col(v["state"]).cast("string"))).alias("state"),
    F.trim(F.col(v["destination"]).cast("string")).alias("destination"),
    F.initcap(F.trim(F.col(v["visitor_type"]).cast("string"))).alias("visitor_type"),
    F.col(v["visitor_count"]).cast("long").alias("visitor_count")
).filter(
    F.col("visit_date").isNotNull() &
    F.col("state").isNotNull() &
    F.col("destination").isNotNull() &
    F.col("visitor_count").isNotNull() &
    (F.col("visitor_count") >= 0)
).dropDuplicates(["visit_date", "state", "destination", "visitor_type"]))

w = COLUMN_MAP["weather"]
require_columns(weather_raw, w, "weather")
weather = (weather_raw.select(
    F.to_date(F.col(w["date"])).alias("weather_date"),
    F.initcap(F.trim(F.col(w["state"]).cast("string"))).alias("state"),
    F.col(w["temp_max"]).cast("double").alias("temp_max_c"),
    F.col(w["temp_min"]).cast("double").alias("temp_min_c"),
    F.col(w["precipitation"]).cast("double").alias("precipitation_mm")
).filter(
    F.col("weather_date").isNotNull() & F.col("state").isNotNull()
).dropDuplicates(["weather_date", "state"]))

d = COLUMN_MAP["destinations"]
require_columns(destinations_raw, d, "destinations")
destinations = (destinations_raw.select(
    F.trim(F.col(d["destination"]).cast("string")).alias("destination"),
    F.initcap(F.trim(F.col(d["state"]).cast("string"))).alias("state"),
    F.col(d["latitude"]).cast("double").alias("latitude"),
    F.col(d["longitude"]).cast("double").alias("longitude"),
    F.initcap(F.trim(F.col(d["category"]).cast("string"))).alias("category")
).filter(
    F.col("destination").isNotNull() & F.col("state").isNotNull()
).dropDuplicates(["destination", "state"]))

visits = (visits
    .withColumn("year", F.year("visit_date"))
    .withColumn("month", F.month("visit_date"))
    .withColumn("month_name", F.date_format("visit_date", "MMMM"))
    .withColumn("quarter", F.quarter("visit_date")))

# Enrich visits with destination attributes. Join keys are destination + state.
visits_enriched = visits.join(destinations, ["destination", "state"], "left")

# Weather join at daily state level. This is appropriate for the sample's daily grain.
# If your tourism data is monthly/annual, aggregate weather to the same grain first.
silver = (visits_enriched
    .join(weather,
          (visits_enriched.visit_date == weather.weather_date) &
          (visits_enriched.state == weather.state),
          "left")
    .drop("weather_date")
    .withColumn("_processed_at", F.current_timestamp()))

(silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{TABLE_BASE}.silver_tourism"))

(destinations.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{TABLE_BASE}.silver_destinations"))

print("Silver tourism rows:", silver.count())
display(silver.orderBy("visit_date", "state", "destination").limit(20))

display(silver.select(
    F.count("*").alias("rows"),
    F.countDistinct("state").alias("states"),
    F.countDistinct("destination").alias("destinations"),
    F.min("visit_date").alias("start_date"),
    F.max("visit_date").alias("end_date"),
    F.sum(F.col("visitor_count").isNull().cast("int")).alias("missing_visitors"),
    F.sum(F.col("temp_max_c").isNull().cast("int")).alias("missing_weather")
))


## 03 Gold Analytics


In [ ]:
# Databricks notebook source

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "tourism_analytics"
TABLE_BASE = f"{CATALOG}.{SCHEMA}"

silver = spark.table(f"{TABLE_BASE}.silver_tourism")

# Destination popularity and visitor KPIs
gold_destination = (silver.groupBy("state", "destination", "category")
    .agg(
        F.sum("visitor_count").alias("total_visitors"),
        F.avg("visitor_count").alias("avg_visitors_per_record"),
        F.max("visitor_count").alias("max_visitors_in_period"),
        F.countDistinct("visit_date").alias("days_reported"),
        F.avg("temp_max_c").alias("avg_max_temperature_c"),
        F.avg("precipitation_mm").alias("avg_precipitation_mm")
    ))

# Monthly seasonal trend
gold_monthly = (silver.groupBy("year", "month", "month_name")
    .agg(
        F.sum("visitor_count").alias("total_visitors"),
        F.countDistinct("destination").alias("destinations_reported"),
        F.avg("temp_max_c").alias("avg_max_temperature_c"),
        F.avg("precipitation_mm").alias("avg_precipitation_mm")
    ))

# Domestic vs foreign visitors
gold_visitor_type = (silver.groupBy("state", "destination", "year", "visitor_type")
    .agg(
        F.sum("visitor_count").alias("total_visitors"),
        F.avg("visitor_count").alias("avg_visitors_per_record")
    ))

# Weather and tourism at daily destination level
gold_weather = (silver.groupBy(
        "visit_date", "year", "month", "state", "destination"
    ).agg(
        F.sum("visitor_count").alias("total_visitors"),
        F.avg("temp_max_c").alias("temp_max_c"),
        F.avg("temp_min_c").alias("temp_min_c"),
        F.avg("precipitation_mm").alias("precipitation_mm")
    ))

# State overview
gold_state = (silver.groupBy("state")
    .agg(
        F.sum("visitor_count").alias("total_visitors"),
        F.countDistinct("destination").alias("destinations_count"),
        F.countDistinct("visit_date").alias("days_reported"),
        F.avg("visitor_count").alias("avg_visitors_per_record")
    ))

tables = {
    "gold_destination_popularity": gold_destination,
    "gold_monthly_trends": gold_monthly,
    "gold_visitor_type": gold_visitor_type,
    "gold_weather_tourism": gold_weather,
    "gold_state_overview": gold_state
}

for name, df in tables.items():
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{TABLE_BASE}.{name}"))
    print(name, "rows:", df.count())
    display(df.limit(10))


## 04 Validation And Export


In [ ]:
# Databricks notebook source

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "tourism_analytics"
TABLE_BASE = f"{CATALOG}.{SCHEMA}"
EXPORT_BASE = "/Volumes/workspace/tourism_analytics/tourism_raw/powerbi_exports"

gold_tables = [
    "gold_destination_popularity",
    "gold_monthly_trends",
    "gold_visitor_type",
    "gold_weather_tourism",
    "gold_state_overview"
]

for table_name in gold_tables:
    df = spark.table(f"{TABLE_BASE}.{table_name}")
    count = df.count()
    print(f"{table_name}: {count} rows")
    if count == 0:
        raise ValueError(f"{table_name} is empty")
    display(df.limit(5))

# Export as CSV folders. Download the part-*.csv file within each folder.
for table_name in gold_tables:
    (spark.table(f"{TABLE_BASE}.{table_name}")
        .coalesce(1)
        .write.mode("overwrite")
        .option("header", True)
        .csv(f"{EXPORT_BASE}/{table_name}"))

print("All Gold tables validated and exported.")
